In this notebook, I verify the desired properties of tje extended transformer model I generate

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

import torch

from extend_model import DecodeExtendedTokenizer, extend_model

In [2]:
MODEL_NAME = "Qwen/Qwen3.5-9B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
extended_tokenizer = DecodeExtendedTokenizer(
    base_tokenizer = tokenizer,
    new_token_strs = ["blackmail"]
)

In [3]:
vocab_size = len(tokenizer)

# Test 1: no change to encoding

input = "blackmail"
print(tokenizer.encode(input))
print(extended_tokenizer.encode(input))

# Test 2: no change to decoding for general token

random_id = random.randint(0, len(tokenizer) - 1)

print(tokenizer.decode([random_id]))
print(extended_tokenizer.decode([random_id]))

# Test 3: able to decode new tokens

new_id = len(tokenizer)

print(extended_tokenizer.decode([new_id]))

[11124, 3585]
[11124, 3585]
糖果
糖果
blackmail


In [4]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

In [5]:
model.eval()

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    out = model(input_ids)

logits = out.logits  # [batch, seq_len, vocab_size]
print(logits.shape)

torch.Size([1, 5, 248320])


In [6]:
new_weights = torch.zeros((2, 4096), dtype=torch.bfloat16)

extend_model(model, tokenizer, new_weights)


Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248079, 4096)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (in_proj_qkv): Linear(in_features=4096, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=4096, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=4096, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=4096, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=

In [7]:
model.eval()

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    out = model(input_ids)

logits = out.logits  # [batch, seq_len, vocab_size]
print(logits.shape)

torch.Size([1, 5, 248079])
